In [807]:
import ssl
import os
import importlib
import json

import torch
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

from transformers import get_scheduler

from utils import set_seed

# Configure SSL for certain operations
ssl._create_default_https_context = ssl._create_unverified_context

import my_datasets
from my_datasets import EmbeddingDataset
from my_datasets import PreSavedBatchDataset
from my_datasets import MNIST
from my_datasets import VisualGenome
import importlib
import trainers

from models import VQELAgent
from trainers import train_self_play, train_mutual_play
from utils import load_dataset
from encoders import ProtoNetCNNEncoder

import torch.nn.functional as F
from torch.distributions import Categorical
from tqdm.auto import tqdm

from test_time import test_time_scaling
from test_time import test_time_sample_adaptation
from test_time import test_time_batch_adaptation
from test_time import test_time_dataset_adaptation
from test_time import oracle_dataset_adaptation


import encoders
import pandas as pd
from collections import defaultdict
from utils import compute_corrects

from torch.utils.data import Dataset
import random
from torch.utils.data import Sampler

from itertools import islice
from tqdm import tqdm
import utils
import my_datasets
from my_datasets import COCO
from models import BaselineAgent
from samplers import OneClassBatchSampler, DogClassBatchSampler, PreSavedBatchSampler

## Config

In [808]:
config_path = os.getenv("CONFIG", "config.py")

if config_path and os.path.exists(config_path):
    spec = importlib.util.spec_from_file_location("config", config_path)
    CFG = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(CFG)
else:
    print("No config file found.")

In [809]:
from dataclasses import dataclass
from typing import Optional, Any
from datetime import datetime


@dataclass
class VQExperimentConfig:
    """Configuration class for VQEL experiment."""

    metric_path = CFG.metric_path
    gumbel = CFG.gumbel
    tau_0 = CFG.tau_0
    baseline = CFG.baseline
    dataset_tt = CFG.dataset_tt
    num_iterations = CFG.num_iterations
    learning_rate_tt = CFG.learning_rate_tt
    message_length_tt = CFG.message_length_tt
    test_time_mode = CFG.test_time_mode
    sampling_temperature_tt = CFG.sampling_temperature_tt
    best_of_n = CFG.best_of_n

    pretrained_checkpoint_a = CFG.pretrained_checkpoint_a
    pretrained_checkpoint_b = CFG.pretrained_checkpoint_b
    dialogued_checkpoint = CFG.dialogued_checkpoint
    sim: str = CFG.sim
    dataset: str = CFG.dataset
    num_test_classes = CFG.num_test_classes

    batch_size: int = CFG.batch_size
    vocab_size: int = CFG.vocab_size
    representation_dim: int = CFG.representation_dim
    input_dim: int = 40
    message_length = CFG.message_length

    # VQ-specific hyperparameters
    ema_dead_code_factor = 2
    threshold_ema_dead_code: float = (
        None  # batch_size / (vocab_size * ema_dead_code_factor)
    )
    decay: float = CFG.decay
    commitment_weight: float = CFG.commitment_weight

    learning_rate_phase1: float = CFG.learning_rate_phase1
    learning_rate_phase2_a: float = CFG.learning_rate_phase2_a
    learning_rate_phase2_b: float = CFG.learning_rate_phase2_b

    weight_decay: float = 1e-5
    num_warmup_steps: int = 100

    num_pretrain_epochs: int = CFG.num_pretrain_epochs

    num_dialogue_epochs: int = CFG.num_dialogue_epochs
    sampling_temperature: float = CFG.sampling_temperature
    entropy_regularization_factor: float = CFG.entropy_regularization_factor
    contrastive_loss_temperature: float = CFG.contrastive_loss_temperature
    agent_a_training_mode: str = (
        CFG.agent_a_training_mode
    )  # Options: 'frozen', 'reinforce_only', 'reinforce_with_preservation'
    freeze_codebook: bool = (
        CFG.freeze_codebook
    )  # Whether to freeze the codebook during text generation

    train_split: float = 0.8
    val_split: float = 0.1
    test_split: float = 0.1

    num_workers: int = 0
    drop_last: bool = True

    # Evaluation parameters
    number_of_candidates: int = CFG.number_of_candidates

    seed: int = CFG.seed
    device: str = "auto"  # Will be set to 'cuda' or 'cpu' based on availability
    ckpt_dir: Optional[str] = None  # Will be generated from hyperparameters
    ckpt_dir_dialogue: Optional[str] = None  # Will be generated from hyperparameters

    # Logging parameters
    log_level: str = "INFO"
    tensorboard_log_dir: Optional[str] = None  # Will be generated from hyperparameters

    def __post_init__(self):
        """Post-initialization to set device and validate splits."""
        self.threshold_ema_dead_code: float = self.batch_size / (
            self.vocab_size * self.ema_dead_code_factor
        )

        if self.device == "auto":
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # Validate that splits sum to 1.0
        total_split = self.train_split + self.val_split + self.test_split
        if abs(total_split - 1.0) > 1e-6:
            raise ValueError(f"Data splits must sum to 1.0, got {total_split}")

        valid_modes = ["frozen", "reinforce_only", "reinforce_with_preservation"]
        if self.agent_a_training_mode not in valid_modes:
            raise ValueError(
                f"agent_a_training_mode must be one of {valid_modes}, got {self.agent_a_training_mode}"
            )

        valid_datasets = ["shape1", "mnist1", "imagenet", "coco", "genome"]
        if self.dataset not in valid_datasets:
            raise ValueError(
                f"dataset must be one of {valid_datasets}, got {self.dataset}"
            )

        # Generate checkpoint directories from hyperparameters
        if self.ckpt_dir is None or self.ckpt_dir_dialogue is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M")
            base_run_name = (
                f"{timestamp}_bs{self.batch_size}_vocab{self.vocab_size}_"
                f"repr{self.representation_dim}_msg_len{self.message_length_tt}_msg_len_tt{self.message_length_tt}_"
                f"lr1_{self.learning_rate_phase1}_lr2a_{self.learning_rate_phase2_a}_lr2b_{self.learning_rate_phase2_b}_"
                f"decay{self.decay}_mode{self.agent_a_training_mode}_"
                f"temp{self.sampling_temperature}_ent{self.entropy_regularization_factor}_"
                f"cand{self.number_of_candidates}_"
                f"contr{self.contrastive_loss_temperature}_seed{self.seed}"
            )

            if self.ckpt_dir is None:
                self.ckpt_dir = f"runs/{base_run_name}/checkpoints/pretraining"

            if self.ckpt_dir_dialogue is None:
                self.ckpt_dir_dialogue = f"runs/{base_run_name}/checkpoints/dialogue"

        if self.tensorboard_log_dir is None:
            base_path = self.ckpt_dir.replace("/pretraining", "").replace(
                "/checkpoints", ""
            )
            self.tensorboard_log_dir = f"{base_path}/tensorboard"

    def to_dict(self) -> dict[str, Any]:
        """Convert config to dictionary."""
        return {
            "gumbel": self.gumbel,
            "tau_0": self.tau_0,
            "baseline": self.baseline,
            "num_test_classes": self.num_test_classes,
            "dataset_tt": self.dataset_tt,
            "test_time_mode": self.test_time_mode,
            "sampling_temperature_tt": self.sampling_temperature_tt,
            "best_of_n": self.best_of_n,
            "num_iterations": self.num_iterations,
            "learning_rate_tt": self.learning_rate_tt,
            "message_length_tt": self.message_length_tt,
            "pretrained_checkpoint_a": self.pretrained_checkpoint_a,
            "pretrained_checkpoint_b": self.pretrained_checkpoint_b,
            "dialogued_checkpoint": self.dialogued_checkpoint,
            "sim": self.sim,
            "dataset": self.dataset,
            "batch_size": self.batch_size,
            "vocab_size": self.vocab_size,
            "representation_dim": self.representation_dim,
            "input_dim": self.input_dim,
            "message_length": f"{self.message_length}",
            "threshold_ema_dead_code": self.threshold_ema_dead_code,
            "decay": self.decay,
            "commitment_weight": self.commitment_weight,
            "learning_rate_phase1": self.learning_rate_phase1,
            "learning_rate_phase2_a": self.learning_rate_phase2_a,
            "learning_rate_phase2_b": self.learning_rate_phase2_b,
            "weight_decay": self.weight_decay,
            "num_warmup_steps": self.num_warmup_steps,
            "num_pretrain_epochs": self.num_pretrain_epochs,
            "num_dialogue_epochs": self.num_dialogue_epochs,
            "sampling_temperature": self.sampling_temperature,
            "entropy_regularization_factor": self.entropy_regularization_factor,
            "contrastive_loss_temperature": self.contrastive_loss_temperature,
            "agent_a_training_mode": self.agent_a_training_mode,
            "freeze_codebook": self.freeze_codebook,
            "train_split": self.train_split,
            "val_split": self.val_split,
            "test_split": self.test_split,
            "num_workers": self.num_workers,
            "drop_last": self.drop_last,
            "number_of_candidates": self.number_of_candidates,
            "seed": self.seed,
            "device": self.device,
            "ckpt_dir": self.ckpt_dir,
            "ckpt_dir_dialogue": self.ckpt_dir_dialogue,
            "log_level": self.log_level,
            "tensorboard_log_dir": self.tensorboard_log_dir,
        }


config = VQExperimentConfig()
device = torch.device(config.device)
print(f"VQEL experiment configuration created: {config}")

VQEL experiment configuration created: VQExperimentConfig(sim='cosine', dataset='genome', batch_size=32, vocab_size=10, representation_dim=2048, input_dim=40, threshold_ema_dead_code=1.6, decay=0.99, commitment_weight=0.25, learning_rate_phase1=1e-05, learning_rate_phase2_a=1e-05, learning_rate_phase2_b=1e-05, weight_decay=1e-05, num_warmup_steps=100, num_pretrain_epochs=50, num_dialogue_epochs=50, sampling_temperature=1e-05, entropy_regularization_factor=0, contrastive_loss_temperature=0.01, agent_a_training_mode='frozen', freeze_codebook=True, train_split=0.8, val_split=0.1, test_split=0.1, num_workers=0, drop_last=True, number_of_candidates=100, seed=1, device='cuda', ckpt_dir='runs/20260925_1427_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_1e-05_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1/checkpoints/pretraining', ckpt_dir_dialogue='runs/20260925_1427_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_1e-05_lr2a_1e-05_lr2b_1e-05_decay0.99_modefro

# Dataset

In [810]:
from generate_datasets.create_mnist import create_mnist1, create_mnist2
from huggingface_hub import snapshot_download
from dotenv import load_dotenv
import os

load_dotenv()

DATA_PATH = os.getenv("DATA_PATH", "../data")
os.makedirs(DATA_PATH, exist_ok=True)

snapshot_download(
    repo_id="MehdiJmlkh/COCO-DINOv2-embeddings",
    repo_type="dataset",
    local_dir=f"{DATA_PATH}/coco",
    local_dir_use_symlinks=False,  
)

snapshot_download(
    repo_id="MehdiJmlkh/ImageNet-ResNet50-embeddings",
    repo_type="dataset",
    local_dir=f"{DATA_PATH}/imagenet",
    local_dir_use_symlinks=False,  
)

snapshot_download(
    repo_id="MehdiJmlkh/ShapeWorld",
    repo_type="dataset",
    local_dir=f"{DATA_PATH}/shape",
    local_dir_use_symlinks=False,  
)

create_mnist1(DATA_PATH)
create_mnist2(DATA_PATH)

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

[SKIP] MNIST1 already exists in ../data/MNIST1
[SKIP] MNIST2 already exists in ../data/MNIST2


In [811]:
# Initialize random seed for reproducibility
set_seed(config.seed)
batch_sampler = None
collate_fn = None


if config.dataset == "shape1":

    train_dataset = load_dataset(f"{DATA_PATH}/shape/train")
    val_dataset = load_dataset(f"{DATA_PATH}/shape/val")
    test_dataset = load_dataset(f"{DATA_PATH}/shape/test/one_shape")

    if config.dataset_tt == "shape1":
        test_time_dataset = load_dataset(f"{DATA_PATH}/shape/test/one_shape")
    else:
        test_time_dataset = load_dataset(f"{DATA_PATH}/shape/test/two_shape")

    object_encoder_a = ProtoNetCNNEncoder()
    object_encoder_b = ProtoNetCNNEncoder()

elif config.dataset == "mnist1":

    dataset = MNIST("train", path=f"{DATA_PATH}/MNIST1")

    test_dataset = random_split(
        MNIST("test", path=f"{DATA_PATH}/MNIST1"), [2000, 8000]
    )[0]
    train_dataset, val_dataset = random_split(dataset, [50000, 10000])

    if config.dataset_tt == "mnist1":
        test_time_dataset = random_split(
            MNIST("test", path=f"{DATA_PATH}/MNIST1"), [2000, 8000]
        )[0]
    else:
        test_time_dataset = random_split(
            MNIST("test", path=f"{DATA_PATH}/MNIST2"), [2000, 8000]
        )[0]

    object_encoder_a = ProtoNetCNNEncoder(in_channels=1)
    object_encoder_b = ProtoNetCNNEncoder(in_channels=1)

elif config.dataset == "imagenet":
    TEST_SAMPLES_PER_CLASS = 32

    data = torch.load(
        f"{DATA_PATH}/imagenet/imagenet_embeddings.pt", map_location="cpu"
    )

    embeddings = data["embeddings"]  # (N, 2048)
    labels = data["labels"]  # (N,)

    df_emb = pd.DataFrame({"idx": range(len(labels)), "label": labels.numpy()})

    test_idx = (
        df_emb.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(n=TEST_SAMPLES_PER_CLASS))["idx"]
        .values
    )
    train_idx = df_emb.drop(test_idx).idx.values

    test_time_dataset = EmbeddingDataset(embeddings, labels, test_idx)

    class_to_indices = defaultdict(list)
    for i, label in enumerate(test_time_dataset.labels.tolist()):
        class_to_indices[label].append(i)

    if config.dataset_tt == "imagenet_dog_breed":
        batch_sampler = DogClassBatchSampler(
            class_to_indices,
            shuffle=False,
            num_classes=config.num_test_classes,
            batch_size=config.number_of_candidates,
        )
    else:
        batch_sampler = OneClassBatchSampler(
            class_to_indices, shuffle=False, num_classes=config.num_test_classes
        )

    dataset = EmbeddingDataset(embeddings, labels, train_idx)
    train_dataset, val_dataset, test_dataset = random_split(dataset, [0.8, 0.1, 0.1])

    object_encoder_a = encoders.IdentityEncoder()
    object_encoder_b = encoders.IdentityEncoder()

    if config.dataset_tt == "imagenet":
        test_time_dataset = test_dataset
        batch_sampler = None

elif config.dataset == "coco":

    train_dataset = COCO(DATA_PATH, "train")
    val_dataset = COCO(DATA_PATH, "val")
    test_dataset = COCO(DATA_PATH, "test")

    if config.dataset_tt == "coco_complex":
        test_time_dataset = COCO(DATA_PATH, "test_similar_batches")

        batch_sampler = PreSavedBatchSampler(len(test_time_dataset))
        collate_fn = lambda x: x[0]
    else:
        test_time_dataset = COCO(DATA_PATH, "test")

    object_encoder_a = encoders.IdentityEncoder(
        representation_dim=config.representation_dim
    )
    object_encoder_b = encoders.IdentityEncoder(
        representation_dim=config.representation_dim
    )

elif config.dataset == "genome":
    dataset = VisualGenome(split="id")
    train_dataset, val_dataset, test_dataset = random_split(dataset, [0.8, 0.1, 0.1])

    if config.dataset_tt == "genome_complex":
        test_time_dataset = VisualGenome(split="ood")
    elif config.dataset_tt == "genome_complex_batch":
        test_time_dataset = VisualGenome(split="ood_similar_batches")
        batch_sampler = PreSavedBatchSampler(len(test_time_dataset))
        collate_fn = lambda x: x[0]
    else:
        test_time_dataset = test_dataset

    object_encoder_a = encoders.IdentityEncoder()
    object_encoder_b = encoders.IdentityEncoder()

In [812]:
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    drop_last=config.drop_last,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    drop_last=config.drop_last,
)

# Agents

In [813]:
if config.baseline:
    agent_a = BaselineAgent(
        input_dim=config.input_dim,
        representation_dim=config.representation_dim,
        vocab_size=config.vocab_size,
        object_encoder=object_encoder_a,
        gumbel=config.gumbel,
        tau_0=config.tau_0,
    ).to(device)
    agent_b = BaselineAgent(
        input_dim=config.input_dim,
        representation_dim=config.representation_dim,
        vocab_size=config.vocab_size,
        object_encoder=object_encoder_b,
        gumbel=config.gumbel,
        tau_0=config.tau_0,
    ).to(device)
else:
    agent_a = VQELAgent(
        input_dim=config.input_dim,
        representation_dim=config.representation_dim,
        threshold_ema_dead_code=config.threshold_ema_dead_code,
        vocab_size=config.vocab_size,
        object_encoder=object_encoder_a,
        decay=config.decay,
        commitment_weight=config.commitment_weight,
        orthogonal_reg_weight=0,
        use_cosine_sim=(config.sim == "cosine"),
    ).to(device)

    agent_b = VQELAgent(
        input_dim=config.input_dim,
        representation_dim=config.representation_dim,
        threshold_ema_dead_code=config.threshold_ema_dead_code,
        vocab_size=config.vocab_size,
        object_encoder=object_encoder_b,
        use_cosine_sim=(config.sim == "cosine"),
    ).to(device)

In [814]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

writer = SummaryWriter(log_dir=config.tensorboard_log_dir)
logger.info(f"TensorBoard writer created at: {config.tensorboard_log_dir}")
logger.info(f"Checkpoint directory (Phase 1): {config.ckpt_dir}")
logger.info(f"Checkpoint directory (Phase 2): {config.ckpt_dir_dialogue}")


2026-09-25 14:27:44,708 [INFO] TensorBoard writer created at: runs/20260925_1427_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_1e-05_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1/tensorboard
2026-09-25 14:27:44,709 [INFO] Checkpoint directory (Phase 1): runs/20260925_1427_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_1e-05_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1/checkpoints/pretraining
2026-09-25 14:27:44,709 [INFO] Checkpoint directory (Phase 2): runs/20260925_1427_bs32_vocab10_repr2048_msg_len4_msg_len_tt4_lr1_1e-05_lr2a_1e-05_lr2b_1e-05_decay0.99_modefrozen_temp1e-05_ent0_cand100_contr0.01_seed1/checkpoints/dialogue


# Self-Play

In [815]:
if config.pretrained_checkpoint_b != None:
    agent_b.load_state_dict(
        torch.load(
            f"../experiments/{config.pretrained_checkpoint_b}/checkpoints/pretraining/best_model.pth",
            weights_only=True,
            map_location=config.device,
        )["agent"]
    )

if config.pretrained_checkpoint_a != None:
    agent_a.load_state_dict(
        torch.load(
            f"../experiments/{config.pretrained_checkpoint_a}/checkpoints/pretraining/best_model.pth",
            weights_only=True,
            map_location=config.device,
        )["agent"]
    )

if (
    (config.pretrained_checkpoint_a == None)
    and (config.pretrained_checkpoint_b == None)
    and (not config.baseline)
):
    optimizer = torch.optim.Adam(
        agent_a.parameters(),
        lr=config.learning_rate_phase1,
        weight_decay=config.weight_decay,
    )

    lr_scheduler = get_scheduler(
        "constant_with_warmup",
        optimizer=optimizer,
        num_warmup_steps=config.num_warmup_steps,
    )

    trainers.train_self_play(
        agent_a,
        train_loader,
        val_loader,
        optimizer,
        lr_scheduler,
        device,
        config.num_pretrain_epochs,
        length_message=config.message_length,
        sampling_temperature=config.sampling_temperature,
        entropy_factor=int(config.entropy_regularization_factor),
        contrastive_loss_temperature=config.contrastive_loss_temperature,
        ckpt_dir=config.ckpt_dir,
    )

In [816]:
from utils import evaluate_self_communicate

if config.baseline:
    self_play_acc_a = 0.0
else:
    self_play_acc_a = evaluate_self_communicate(
        agent_a,
        test_dataset,
        device,
        config.message_length[-1],
        number_of_candidates=config.number_of_candidates,
    )

Evaluating image-text matching: 100%|██████████| 15/15 [00:00<00:00, 140.23it/s]

Image-text matching accuracy: 0.553


In [817]:
if config.baseline:
    self_play_acc_b = 0.0
else:
    self_play_acc_b = evaluate_self_communicate(
        agent_b,
        test_dataset,
        device,
        config.message_length[-1],
        number_of_candidates=config.number_of_candidates,
    )

Evaluating image-text matching: 100%|██████████| 15/15 [00:00<00:00, 144.93it/s]

Image-text matching accuracy: 0.009


# Mutual Play 

In [818]:
optimizer = torch.optim.Adam(
    [
        {
            "params": agent_a.parameters(),
            "lr": config.learning_rate_phase2_a,
            "weight_decay": config.weight_decay,
        },
        {
            "params": agent_b.parameters(),
            "lr": config.learning_rate_phase2_b,
            "weight_decay": config.weight_decay,
        },
    ]
)

num_training_steps = config.num_pretrain_epochs * len(train_loader)

lr_scheduler = get_scheduler(
    "constant_with_warmup",
    optimizer=optimizer,
    num_warmup_steps=config.num_warmup_steps,
)

In [819]:
if config.dialogued_checkpoint != None:
    agent_a.load_state_dict(
        torch.load(
            f"../experiments/{config.dialogued_checkpoint}/checkpoints/dialogue/best_model.pth",
            weights_only=True,
            map_location=config.device,
        )["agent_a"]
    )
    agent_b.load_state_dict(
        torch.load(
            f"../experiments/{config.dialogued_checkpoint}/checkpoints/dialogue/best_model.pth",
            weights_only=True,
            map_location=config.device,
        )["agent_b"]
    )

elif config.baseline:
    best_val_acc, best_model_state = trainers.train_agents_baseline(
        agent_a=agent_a,
        agent_b=agent_b,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        lr_scheduler=lr_scheduler,
        device=device,
        num_epochs=config.num_dialogue_epochs,
        message_length=config.message_length,
        sampling_temperature=config.sampling_temperature,
        entropy_regularization_factor=config.entropy_regularization_factor,
        contrastive_loss_temperature=config.contrastive_loss_temperature,
        ckpt_dir=config.ckpt_dir_dialogue,
        tensorboard_writer=writer,
        logger=logging.getLogger(__name__),
        gumbel=config.gumbel,
    )

    agent_a.load_state_dict(
        torch.load(f"./{config.ckpt_dir_dialogue}/best_model.pth", weights_only=True)[
            "agent_a"
        ]
    )
    agent_b.load_state_dict(
        torch.load(f"./{config.ckpt_dir_dialogue}/best_model.pth", weights_only=True)[
            "agent_b"
        ]
    )

else:
    best_model_state, best_val_acc = trainers.train_mutual_play(
        agent_a,
        agent_b,
        train_loader,
        val_loader,
        device,
        config.message_length,
        config.num_pretrain_epochs,
        optimizer,
        lr_scheduler,
        num_dialogue_epochs=config.num_dialogue_epochs,
        sampling_temperature=config.sampling_temperature,
        entropy_regularization_factor=config.entropy_regularization_factor,
        contrastive_loss_temperature=config.contrastive_loss_temperature,
        ckpt_dir=config.ckpt_dir_dialogue,
        agent_a_training_mode=config.agent_a_training_mode,
        freeze_codebook=config.freeze_codebook,
        tensorboard_writer=writer,
    )

    agent_a.load_state_dict(
        torch.load(f"./{config.ckpt_dir_dialogue}/best_model.pth", weights_only=True)[
            "agent_a"
        ]
    )
    agent_b.load_state_dict(
        torch.load(f"./{config.ckpt_dir_dialogue}/best_model.pth", weights_only=True)[
            "agent_b"
        ]
    )

In [820]:
from utils import evaluate_cross_communicate

mutual_play_acc = evaluate_cross_communicate(
    agent_a,
    agent_b,
    test_dataset,
    device,
    config.message_length[-1],
    config.number_of_candidates,
)
print(mutual_play_acc)

Evaluating: 100%|██████████| 15/15 [00:00<00:00, 142.84it/s]

0.618


# Test time Computation

In [821]:
# Accuracy on OOD without test time training or adaptation
utils.evaluate_cross_communicate(
    agent_a,
    agent_b,
    test_time_dataset,
    device,
    config.message_length_tt,
    config.number_of_candidates,
    batch_sampler=batch_sampler,
    collate_fn=collate_fn,
)

Evaluating: 100%|██████████| 15/15 [00:00<00:00, 151.69it/s]


0.017333333333333333

In [822]:
test_loader = DataLoader(
    test_dataset,
    batch_size=100,
    shuffle=False,
    num_workers=1,
)

In [823]:
test_time_loader = DataLoader(
    test_time_dataset, batch_sampler=batch_sampler, collate_fn=collate_fn
)

In [824]:
for imgs, lables in test_loader:
    print(imgs)

tensor([[0.0313, 0.0000, 0.0000,  ..., 0.0000, 0.0091, 0.0000],
        [0.2238, 0.0452, 0.0674,  ..., 0.0000, 0.0324, 0.7558],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0341],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.5794, 0.0000, 0.0000],
        [0.0562, 0.0000, 0.1773,  ..., 1.3558, 0.0000, 0.0567],
        [0.0000, 0.4318, 0.0000,  ..., 0.0000, 0.1770, 0.0000]])
tensor([[5.3127e-03, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         1.6916e-01],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 5.3827e-03, 0.0000e+00,
         0.0000e+00],
        [1.1296e+00, 0.0000e+00, 1.5071e-01,  ..., 4.5965e-04, 0.0000e+00,
         4.5633e-02],
        ...,
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 2.9780e-01, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 1.2315e-01,  ..., 0.0000e+00, 0.0000e+00,
         2.3913e-04],
        [1.9535e-02, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         9.2303e-02]])
tensor

In [825]:
for imgs, lables in test_time_loader:
    print(imgs)

tensor([[0.0048, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0023, 0.0003,  ..., 0.0009, 0.0000, 0.0006],
        [0.0008, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0149, 0.0000,  ..., 0.0000, 0.0000, 0.0005],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0010]])
tensor([[0.0000, 0.0000, 0.0042,  ..., 0.0035, 0.0000, 0.0000],
        [0.0449, 0.0020, 0.0120,  ..., 0.0134, 0.0028, 0.0047],
        [0.0040, 0.0000, 0.0185,  ..., 0.0017, 0.0000, 0.0239],
        ...,
        [0.0000, 0.0038, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0052, 0.0000, 0.0009,  ..., 0.0007, 0.0028, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])
tensor([[0.0002, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0004, 0.0011, 0.0006,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0091,  ..., 0.0000, 0.0007, 0.0000],
        ...,

In [826]:
if batch_sampler == None:
    test_loader = DataLoader(
        test_time_dataset,
        batch_size=config.number_of_candidates,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn,
    )
else:
    test_loader = DataLoader(
        test_time_dataset, batch_sampler=batch_sampler, collate_fn=collate_fn
    )

In [827]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

num_iterations = len(test_loader)
exclude_starts = [torch.cuda.Event(enable_timing=True) for _ in range(num_iterations)]
exclude_ends = [torch.cuda.Event(enable_timing=True) for _ in range(num_iterations)]

In [828]:
start_event.record()

if (
    config.test_time_mode == "oracle_adaptation"
    or config.test_time_mode == "oracle_full_adaptation"
):
    if config.test_time_mode == "oracle_full_adaptation":
        optimizer = torch.optim.Adam(
            list(agent_a.parameters()) + list(agent_b.parameters()),
            lr=config.learning_rate_tt,
        )
    else:
        optimizer = torch.optim.Adam(
            [
                {"params": agent_a.text_generation_gru.parameters()},
                {"params": agent_a.text_generation_gru_head.parameters()},
            ],
            lr=config.learning_rate_tt,
        )

    lr_scheduler = get_scheduler(
        "constant",
        optimizer=optimizer,
    )

    ttc_mutual_acc, best_model_state = trainers.train_agents_baseline(
        agent_a=agent_a,
        agent_b=agent_b,
        train_loader=test_loader,
        val_loader=test_loader,
        optimizer=optimizer,
        lr_scheduler=lr_scheduler,
        device=device,
        num_epochs=config.num_iterations,
        message_length=[config.message_length_tt],
        sampling_temperature=config.sampling_temperature_tt,
        entropy_regularization_factor=config.entropy_regularization_factor,
        contrastive_loss_temperature=config.contrastive_loss_temperature,
        ckpt_dir=config.ckpt_dir_dialogue,
        tensorboard_writer=writer,
        logger=logging.getLogger(__name__),
        gumbel=config.gumbel,
    )

    ttc_self_play_acc = 0.0

elif config.baseline:
    ttc_self_play_acc = 0.0
    ttc_mutual_acc = evaluate_cross_communicate(
        agent_a,
        agent_b,
        test_time_dataset,
        device,
        config.message_length_tt,
        config.number_of_candidates,
        batch_sampler=batch_sampler,
        collate_fn=collate_fn,
    )

    end_event.record()
    ttc_peak_mem = torch.cuda.max_memory_allocated(device)

elif config.test_time_mode == "full_sender_dataset_adaptation":
    test_time_dataset_adaptation(
        agent_a,
        test_loader,
        message_length=config.message_length_tt,
        num_epochs=config.num_iterations,
        lr=config.learning_rate_tt,
        sampling_temperature=config.sampling_temperature_tt,
        full_adaptation=True,
    )

    ttc_mutual_acc = evaluate_cross_communicate(
        agent_a,
        agent_b,
        test_time_dataset,
        device,
        config.message_length_tt,
        config.number_of_candidates,
        batch_sampler=batch_sampler,
        collate_fn=collate_fn,
    )

    end_event.record()
    ttc_peak_mem = torch.cuda.max_memory_allocated(device)

    ttc_self_play_acc = evaluate_self_communicate(
        agent_a,
        test_time_dataset,
        device,
        config.message_length_tt,
        number_of_candidates=config.number_of_candidates,
        batch_sampler=batch_sampler,
        collate_fn=collate_fn,
    )

elif config.test_time_mode == "oracle_dataset_adaptation":
    oracle_dataset_adaptation(
        agent_a,
        agent_b,
        test_loader,
        message_length=config.message_length_tt,
        num_epochs=config.num_iterations,
        lr=config.learning_rate_tt,
        sampling_temperature=config.sampling_temperature_tt,
    )

    ttc_mutual_acc = evaluate_cross_communicate(
        agent_a,
        agent_b,
        test_time_dataset,
        device,
        config.message_length_tt,
        config.number_of_candidates,
        batch_sampler=batch_sampler,
        collate_fn=collate_fn,
    )

    end_event.record()
    ttc_peak_mem = torch.cuda.max_memory_allocated(device)

    ttc_self_play_acc = evaluate_self_communicate(
        agent_a,
        test_time_dataset,
        device,
        config.message_length_tt,
        number_of_candidates=config.number_of_candidates,
        batch_sampler=batch_sampler,
        collate_fn=collate_fn,
    )

elif config.test_time_mode == "dataset_adaptation":
    test_time_dataset_adaptation(
        agent_a,
        test_loader,
        message_length=config.message_length_tt,
        num_epochs=config.num_iterations,
        lr=config.learning_rate_tt,
        sampling_temperature=config.sampling_temperature_tt,
    )

    ttc_mutual_acc = evaluate_cross_communicate(
        agent_a,
        agent_b,
        test_time_dataset,
        device,
        config.message_length_tt,
        config.number_of_candidates,
        batch_sampler=batch_sampler,
        collate_fn=collate_fn,
    )

    end_event.record()
    ttc_peak_mem = torch.cuda.max_memory_allocated(device)

    ttc_self_play_acc = evaluate_self_communicate(
        agent_a,
        test_time_dataset,
        device,
        config.message_length_tt,
        number_of_candidates=config.number_of_candidates,
        batch_sampler=batch_sampler,
        collate_fn=collate_fn,
    )

else:
    agent_b.eval()
    total_correct, total_ins = 0, 0
    total_correct_a = 0

    if config.test_time_mode == "sample_adaptation":
        half_len = len(test_loader) // 2
        progress_bar = tqdm(
            islice(test_loader, half_len),
            total=half_len,
            desc="Testing time computation",
        )
    else:
        progress_bar = tqdm(test_loader, desc="Testing time computation")

    for iter_num, (imgs, labels) in enumerate(progress_bar):
        imgs = imgs.to(device)

        if config.test_time_mode == "batch_adaptation":
            words, word_reprs = test_time_batch_adaptation(
                agent_a,
                imgs,
                message_length=config.message_length_tt,
                labels=labels,
                num_epochs=config.num_iterations,
                lr=config.learning_rate_tt,
                sampling_temperature=config.sampling_temperature_tt,
            )
        else:
            ttc_words = []
            ttc_word_reprs = []
            for i in tqdm(range(len(imgs)), desc="Test-time training", leave=False):
                sample = imgs[i]

                if config.test_time_mode == "scaling":
                    ttc_word, ttc_discretized = test_time_scaling(
                        agent_a,
                        sample,
                        n=config.best_of_n,
                        sampling_temperature=config.sampling_temperature_tt,
                        message_length=config.message_length_tt,
                    )
                else:
                    ttc_word, ttc_discretized = test_time_sample_adaptation(
                        agent_a,
                        sample,
                        num_iterations=config.num_iterations,
                        lr=config.learning_rate_tt,
                        sampling_temperature=config.sampling_temperature_tt,
                        length_message=config.message_length_tt,
                    )

                ttc_words.append(ttc_word)
                ttc_word_reprs.append(ttc_discretized)

            words = torch.stack(ttc_words, dim=0).squeeze(1)
            word_reprs = torch.stack(ttc_word_reprs, dim=0).squeeze(1)

        # Evaluate with the listener (agent_b)
        listener_messages_repr = agent_b.forward_external_text_perception(
            words
        ).squeeze(1)
        listener_objects_repr = agent_b.forward_image_encoder(imgs)
        batch_correct, _ = compute_corrects(
            listener_messages_repr, listener_objects_repr, idx=labels
        )
        total_correct += batch_correct

        # Evaulate with the sender (agent_a)
        exclude_starts[
            iter_num
        ].record()  # ---------------------------------------------------------------
        agent_a.eval()
        text_representations_a = agent_a.forward_text_perception(word_reprs)
        image_representations_a = agent_a.forward_image_encoder(imgs)
        corrects, total = compute_corrects(
            text_representations_a, image_representations_a, idx=labels
        )
        total_correct_a += corrects
        total_ins += total

        batch_accuracy = round(batch_correct / len(imgs), 3)
        cumulative_accuracy = round(total_correct / total_ins, 3)
        progress_bar.set_postfix(batch_acc=batch_accuracy, cum_acc=cumulative_accuracy)
        progress_bar.refresh()

        agent_a.train()
        exclude_ends[
            iter_num
        ].record()  # -----------------------------------------------------------------

    ttc_self_play_acc = total_correct_a / total_ins
    ttc_mutual_acc = total_correct / total_ins

    end_event.record()
    ttc_peak_mem = torch.cuda.max_memory_allocated(device)


print("Final self play accuracy:", round(ttc_self_play_acc, 3))
print("Final test accuracy with test-time training:", round(ttc_mutual_acc, 3))

Training Progress Epoch 0/30:   0%|          | 0/15 [00:00<?, ?it/s]

Training Progress Epoch 1/30:   0%|          | 0/15 [00:00<?, ?it/s]

Training Progress Epoch 2/30:   0%|          | 0/15 [00:00<?, ?it/s]

Training Progress Epoch 3/30:   0%|          | 0/15 [00:00<?, ?it/s]

Training Progress Epoch 4/30:   0%|          | 0/15 [00:00<?, ?it/s]

...

Training Progress Epoch 27/30:   0%|          | 0/15 [00:00<?, ?it/s]

Training Progress Epoch 28/30:   0%|          | 0/15 [00:00<?, ?it/s]

Training Progress Epoch 29/30:   0%|          | 0/15 [00:00<?, ?it/s]

Evaluating image-text matching: 100%|██████████| 15/15 [00:00<00:00, 154.66it/s]

Image-text matching accuracy: 0.352
Final self play accuracy: 0.352
Final test accuracy with test-time training: 0.347


# Metrics

## Similarity between MP embeddings 

In [829]:
if config.metric_path != None:
    mp_similarity = utils.evaluate_similarity_between_text_perceptions(
        agent_a,
        agent_b,
        test_dataset,
        device,
        config.message_length[-1],
        config.number_of_candidates,
    )
    mp_similarity

## Time

In [830]:
torch.cuda.synchronize()

if not config.baseline and config.test_time_mode == "batch_full_sender_adaptation":
    excluded_time = sum(
        exclude_starts[i].elapsed_time(exclude_ends[i]) for i in range(num_iterations)
    )
else:
    excluded_time = 0

ttc_inference_time = start_event.elapsed_time(end_event) - excluded_time

print("Inference time: ", ttc_inference_time)

Inference time:  10796.83203125


## Peak Memory

In [831]:
print("Peak memory: ", ttc_peak_mem)

Peak memory:  1317745664


## FLOPS

In [832]:
import torch
import torch.nn as nn
from fvcore.nn import FlopCountAnalysis


class ForwardWrapper_OP(nn.Module):
    def __init__(self, agent_a, message_length):
        super().__init__()
        self.agent_a = agent_a
        self.agent_b = agent_b
        self.message_length = message_length

    def forward(self, imgs):
        generated_output = self.agent_a.forward_image_encoder(imgs)
        return generated_output


class ForwardWrapper(nn.Module):
    def __init__(self, agent_a, message_length):
        super().__init__()
        self.agent_a = agent_a
        self.message_length = message_length

    def forward(self, imgs):
        generated_output = self.agent_a.forward_text_generation(
            imgs,
            message_length=self.message_length,
            freeze_codebook=True,
            mode="discrete",
            sampling_temperature=1e-5,
        )
        words = generated_output["indices"]

        listener_messages_repr = self.agent_a.forward_external_text_perception(
            words
        ).squeeze(1)

        return listener_messages_repr


if config.dataset_tt == "mnist2":
    input_data = torch.randn(config.number_of_candidates, 28, 56, 1).to(config.device)
else:
    input_data = torch.randn(config.number_of_candidates, config.representation_dim).to(
        config.device
    )
if config.metric_path != None:
    flops_OP = FlopCountAnalysis(
        ForwardWrapper_OP(agent_a, config.message_length_tt), input_data
    ).total()
    flops_OP_to_MP = FlopCountAnalysis(
        ForwardWrapper(agent_a, config.message_length_tt), input_data
    ).total()

In [833]:
def calculate_flops(flops_OP, flops_OP_to_MP, num_steps):
    flops_MG_and_MP = flops_OP_to_MP - flops_OP

    tta_flops = num_steps * 3 * flops_MG_and_MP
    inference_flops = flops_OP_to_MP + flops_OP

    return tta_flops + inference_flops


if config.metric_path != None:
    total_flops = calculate_flops(flops_OP, flops_OP_to_MP, config.num_iterations)

# Save

In [834]:
if config.metric_path != None:
    result_dir = f"../experiments/{config.metric_path}/results"

    report_path = os.path.join(result_dir, "report.json")

    with open(report_path, "r") as f:
        results = json.load(f)

    # Update / add the metric
    results["metrics"]["mp_similarity"] = round(mp_similarity["observed_similarity"], 6)
    results["metrics"]["mp_similarity_baseline_mean"] = round(mp_similarity["baseline_mean"], 6)
    results["metrics"]["mp_similarity_baseline_std"] = round(mp_similarity["baseline_std"], 6)
    results["metrics"]["mp_similarity_p_value"] = round(mp_similarity["p_value"], 6)
    results["metrics"]["inference_time"] = ttc_inference_time
    results["metrics"]["peak_memory"] = ttc_peak_mem
    results["metrics"]["flops"] = total_flops


    with open(report_path, "w") as f:
        json.dump(results, f, indent=4)
else:
    result_dir = config.tensorboard_log_dir.replace("tensorboard", "results")
    os.makedirs(result_dir, exist_ok=True)

    results = {
        "VQEL": True,
        "metrics": {
            "self_play_accuracy_a": round(self_play_acc_a, 3),
            "self_play_accuracy_b": round(self_play_acc_b, 3),
            "mutual_play_accuracy": round(mutual_play_acc, 3),
            "test_time_self_play_accuracy": round(ttc_self_play_acc, 3),
            "test_time_training_accuracy": round(ttc_mutual_acc, 3),
        },
        "config": {k: v for k, v in config.to_dict().items()}
    }

    with open(os.path.join(result_dir, "report.json"), "w") as file:
        json.dump(results, file, indent=4)